# Spatio-temporal Hawkes processes

Events now carry a location as well as a time. This notebook uses
`LegacySpatioTemporalHawkesProcess`, which works on a periodic interval; for new
work prefer `SpatioTemporalHawkesProcess`, which supports arbitrary domains
(`Circle`, `Torus2D`, or your own `SpatialDomain`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import hawkes_package as hp

In [ ]:
# the temporal kernel
def HawkesIntensity_temporal(time):
    a = 0.9
    b = 2
    IndInTemp = (time < b / 2) & (time > 0)
    IndDecTemp = (time >= b / 2) & (time < b)
    return 2 * a / b * (time) * IndInTemp + ((-(2 * a) / b) * (time) + 2 * a) * IndDecTemp


# the spatial kernel
def HawkesIntensity_spatial(space):
    #     a = 1.5
    b = np.pi
    Ind = ((space + b / 2) >= 0) & ((space + b / 2) <= b)
    return (504 / (5 * np.pi**4) * space**4 - 146 / (5 * np.pi**2) * space**2 + 1) * Ind


def Base(x):
    mu = 0.5
    return mu


#  -------------- Plotting the kernels
x = np.linspace(0, 2 * np.pi, 100)

plt.subplot(1, 2, 1)
plt.plot(x, HawkesIntensity_temporal(x), "m")
plt.xlabel("time")
plt.ylabel("temporal kernel over time")


plt.subplot(1, 2, 2)
plt.plot(
    x - np.pi, HawkesIntensity_spatial(x - np.pi), "r"
)  # Only for plot reasons, h is symmetric around 0
plt.xlabel("space")
plt.ylabel("spatial kernel on X")

plt.tight_layout()

plt.show()

In [ ]:
G = hp.LegacySpatioTemporalHawkesProcess(
    Base, HawkesIntensity_spatial, HawkesIntensity_temporal, rng=42
)

In [ ]:
# 25 events rather than 100: each one costs a quadrature sweep across space plus
# a 2000-step Metropolis chain, and this notebook runs on every docs build.
G.simulate(25)
print(f"{G.Events.shape[1]} events, last at t = {G.Events[0, -1]:.2f}")

## The intensity field

Before 0.2.0 this section re-implemented the field intensity by hand, because
the class exposed no accessor at all. It now comes straight from the process,
which also guarantees it matches what the simulator actually used.

In [ ]:
time, space, z = G.intensity_over_interval(
    np.linspace(0, G.Events[0, -1], 200),
    points=np.linspace(-np.pi, np.pi, 200),
)
# `z` is (n_space, n_time): rows index space, columns index time.
z.shape

In [ ]:
plt.figure(figsize=(10, 6))
plt.contourf(time, space, z, 50, cmap="jet")
plt.colorbar(label=r"$\lambda(t, x \mid H_t)$")
plt.scatter(G.Events[0, :], G.Events[1, :], c="r", marker="^", label="events")
plt.xlabel("time")
plt.ylabel("space")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

## A domain-aware process

The same idea on an explicit `Circle`, using the modern class. Swapping in
`Torus2D()` moves the simulation to two spatial dimensions with no other change.

In [ ]:
P = hp.SpatioTemporalHawkesProcess(
    base=lambda x: 0.5,
    spatial=lambda d: max(0.0, 1 - d / np.pi),
    temporal=lambda dt: 0.9 * np.exp(-5 * dt),
    domain=hp.Circle(),
    monotone_temporal_kernel=True,
    rng=0,
)
P.simulate(20)

plt.figure(figsize=(9, 3))
plt.scatter(P.Events[0, :], P.Events[1, :], c="r", marker="^")
plt.xlabel("time")
plt.ylabel("position on the circle")
plt.tight_layout()
plt.show()